# 07 结构化输出 🎉

能返回json格式的结果，但阿里云百炼的responses接口不会按照Reasoning schema输出。

Firework AI有structured output支持，参见[docs](https://docs.fireworks.ai/structured-responses/structured-response-formatting)，但只支持chat/completions接口。

<span style="color: #E74C3C">🎉🎉🎉 ZenMux简直是救星，100%支持OpenAI的Responses API接口。</span>

---

OpenAI官网推荐用responses.parse()+text_format来实现结构化输出，但也支持提供json_schema，但这种形式只有[部分模型支持](https://developers.openai.com/api/docs/guides/structured-outputs#structured-outputs-vs-json-mode)：
- gpt-4o-mini
- gpt-4o-mini-2024-07-18
- gpt-4o-2024-08-06
- 以及之后发行的版本

In [2]:
import os
import json
from rich import print
from dotenv import load_dotenv
from semantic_kernel.agents import OpenAIResponsesAgent, AgentThread

In [ ]:
# 完全复刻了OpenAI的Responses API
load_dotenv("../.env/zenmux", override=True)

True

In [19]:
model_id = os.environ["OPENAI_RESPONSES_MODEL_ID"]
model_id

'openai/gpt-5.4-mini'

In [3]:
from pydantic import BaseModel


class Step(BaseModel):
    explanation: str
    output: str


class Solving(BaseModel):
    steps: list[Step]
    final_answer: str

In [21]:
user_ask_math = [
    "how can I solve 8x + 7y = -23, and 4x=12? Reply in JSON format.",
]

In [22]:
client = OpenAIResponsesAgent.create_client()
agent_s = OpenAIResponsesAgent(
    ai_model_id=model_id,
    client=client,
    instructions="Solve math problems.",
    name="StructuredOutputsAgent",
    store_enabled=False,
    # 指定格式
    text=OpenAIResponsesAgent.configure_response_format(Solving),
)

In [4]:
OpenAIResponsesAgent.configure_response_format(Solving)

{'format': {'type': 'json_schema',
  'name': 'Solving',
  'schema': {'$defs': {'Step': {'properties': {'explanation': {'title': 'Explanation',
       'type': 'string'},
      'output': {'title': 'Output', 'type': 'string'}},
     'required': ['explanation', 'output'],
     'title': 'Step',
     'type': 'object',
     'additionalProperties': False}},
   'properties': {'steps': {'items': {'$ref': '#/$defs/Step'},
     'title': 'Steps',
     'type': 'array'},
    'final_answer': {'title': 'Final Answer', 'type': 'string'}},
   'required': ['steps', 'final_answer'],
   'title': 'Solving',
   'type': 'object',
   'additionalProperties': False},
  'strict': True}}

In [25]:
thread: AgentThread = None

for user_input in user_ask_math:
    print(f"[red]User ▶︎ {str(user_input)}[/red]")

    response = await agent_s.get_response(messages=user_input, thread=thread)
    reasoned_result = Solving.model_validate_json(response.message.content)

    pretty_display = json.dumps(reasoned_result.model_dump(), indent=4, ensure_ascii=False)
    print(f"[green]{response.name} ▶︎ \n{pretty_display}[/green]")

    thread = response.thread

User ▶︎ how can I solve 8x + 7y = -23, and 4x=12? Reply in JSON format.

StructuredOutputsAgent ▶︎ 
{
    "steps": [
        {
            "explanation": "Solve the second equation first: 4x = 12, so divide both sides by 4.",
            "output": "x = 3"
        },
        {
            "explanation": "Substitute x = 3 into the first equation: 8x + 7y = -23.",
            "output": "8(3) + 7y = -23"
        },
        {
            "explanation": "Simplify and solve for y.",
            "output": "24 + 7y = -23"
        },
        {
            "explanation": "Subtract 24 from both sides.",
            "output": "7y = -47"
        },
        {
            "explanation": "Divide both sides by 7.",
            "output": "y = -47/7"
        }
    ],
    "final_answer": "The solution is x = 3 and y = -47/7."
}